# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Problem Framing & ML Task Type
For **Lane 2: Refresh Opportunity Scoring**, my goal is to produce a ranked refresh queue that prioritizes pages that show signs of declining performance for review for refresh. This will be a ranking and scoring task evaluated using Precision@50, the proportion of pages that are actually declining in my top 50 rankings


### Model Selection
I have selected 4 methods of increasing complexity to proceed with:

1. **Static Baseline Rule:**
   * **Formula:** page_score = tier_change * visibility_score (where tier_change is the change in ranking position tier between the first week and second week of March 2026, and visibility_score is mapped from impressions from the second week of March).\
   This establishes the baseline that any of my machine learning models must beat.

2. **Logistic Regression:**
   * Standardizes features and learns linear probability weights, which I could print if I wanted. Serves as my simplest machine learning model.

3. **Shallow Decision Tree**
   * Learns a human-readable chain of if/else decisions. I have used a max_depth of 4, meaning there are no more than 4 decisions made along each path from the root to the leaves, to reduce the risk of the model overfitting to the data. This also allows me to manually inspect each of the decisions the tree makes.

4. **Random Forest:**
   * Random Forest combines multiple decision trees to capture non-linear relationships between features such as ranking tier, changes in clicks and impressions. Also, multiple decision trees combined are less likely to make mistakes than a single decision tree.


### Core Principle:
I will prefer a more complex model only when the mean of its Precision@50 values is higher than both the baseline rule and any other less-complex models


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I am grouping by client because otherwise pages that belong to the same client would end up in both the training and testing data. This could result in my models identifying the client that owns each page instead of learning patterns that indicate declining performance. This would be due to pages belonging to the same client sharing things such as technical infrastructure and traffic levels.

Also, having all the data of certain clients in the test data allows me to confirm that my model can generalise and identify declining pages belonging to clients it has never seen before.

### Leakage Control & Feature Boundaries:
* **Decision point: 15th March 2026**

* **Target Label (`future_decline`):** A binary indicator where `1` means the page experienced a ranking tier drop (`tier_change > 0`) between the first week of March (March 1–7) and the second half of March (March 16–31). This will indicate that it the page is showing signs of decline.

* **Included Features:** Features known strictly by March 15th, 2026, capturing Weeks 1 and 2 performance and historical trend changes:
  * `week1_impressions`, `w1_tier`, `week1_avg_position`,
  * `week2_impressions`, `w2_tier`, `week2_avg_position`,
  * `click_change_w1_w2`, `impression_change_w1_w2`, `historical_tier_change`,
  * `log_w2_impressions`, `log_w2_clicks`, `w2_ctr`


* **Excluded Features:** All Late March metrics (`late_clicks`, `late_impressions`, `late_avg_position`, `late_tier`) are excluded from model features because they come from after the decision point (March 15, 2026). They are only used to determine the future target label (`future_decline`).

In [1]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

# 1. Hugging Face Authentication & DuckDB Setup
hf_token = userdata.get("HF_TOKEN") if "google.colab" in sys.modules else os.environ.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
page_performance_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- Week 1 (March 1-7) - Last Week
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_clicks ELSE 0 END) AS week1_clicks,
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_impressions ELSE 0 END) AS week1_impressions,
    AVG(CASE WHEN report_date <= '2026-03-07' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week1_avg_position,

    -- Week 2 (March 8-15) - This Week
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS week2_clicks,
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS week2_impressions,
    AVG(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week2_avg_position,

    -- Late March (March 16-31) - The Future Target Outcome
    SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks,
    SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions,
    AVG(CASE WHEN report_date > '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS late_avg_position

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

df = con.sql(page_performance_query).df()
print(f"Loaded {len(df):,} rows from DuckDB.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 63,856 rows from DuckDB.


In [3]:
def position_tier(pos):
    if pd.isna(pos) or pos == 0:
      return np.nan

    if pos <= 3:
        return 1
    elif pos <= 10:
        return 2
    elif pos <= 20:
        return 3
    elif pos <= 50:
        return 4
    else:
        return 5

df["w1_tier"] = df["week1_avg_position"].apply(position_tier)
df["w2_tier"] = df["week2_avg_position"].apply(position_tier)
df["late_tier"] = df["late_avg_position"].apply(position_tier)

# Historical tier change (Week 1 to Week 2, known by March 15th)
df["historical_tier_change"] = df["w2_tier"] - df["w1_tier"]

df.head()

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,late_clicks,late_impressions,late_avg_position,w1_tier,w2_tier,late_tier,historical_tier_change
0,client_65de48885f4ef01b,content_5c80451459c29b4a,0.0,5.0,5.400000,0.0,0.0,NaN,0.0,0.0,NaN,2.0,NaN,NaN,NaN
1,client_65de48885f4ef01b,content_6b0149a80607dac3,1.0,196.0,8.830638,1.0,269.0,7.282249,10.0,734.0,8.174629,2.0,2.0,2.0,0.0
2,client_65de48885f4ef01b,content_62673eea26c31c17,6.0,24542.0,6.187605,29.0,24844.0,5.392198,8.0,7759.0,7.866490,2.0,2.0,2.0,0.0
3,client_65de48885f4ef01b,content_872342e050545a12,0.0,39.0,6.538462,0.0,0.0,NaN,0.0,0.0,NaN,2.0,NaN,NaN,NaN
4,client_65de48885f4ef01b,content_4c185d1c173cd53d,3.0,113.0,10.719087,3.0,43.0,11.539352,3.0,122.0,9.395247,3.0,3.0,2.0,0.0


In [4]:
def visibility_bucket(impressions):
    if pd.isna(impressions):
        return np.nan
    elif impressions <= 8:
        return "low"
    elif impressions <= 459:
        return "medium"
    else:
        return "high"

df["visibility_bucket"] = df["week2_impressions"].apply(visibility_bucket)
df["visibility_score"] = df["visibility_bucket"].map({"low": 1, "medium": 2, "high": 4})

def calculate_baseline_score(row):
    if pd.notna(row["historical_tier_change"]) and row["historical_tier_change"] > 0:
        return row["historical_tier_change"] * row["visibility_score"]
    else:
        return 0.0

df["baseline_score"] = df.apply(calculate_baseline_score, axis=1)

model_df = df[df["w1_tier"].notna() & df["w2_tier"].notna() & df["late_tier"].notna()].copy().reset_index(drop=True)
model_df["future_decline"] = (model_df["late_tier"] > model_df["w1_tier"]).astype(int)

print(f"Dataset filtered: {len(model_df):,} rows with valid early positions.")

model_df.head()

Dataset filtered: 13,531 rows with valid early positions.


,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,late_clicks,late_impressions,late_avg_position,w1_tier,w2_tier,late_tier,historical_tier_change,visibility_bucket,visibility_score,baseline_score,future_decline
0,client_65de48885f4ef01b,content_6b0149a80607dac3,1.0,196.0,8.830638,1.0,269.0,7.282249,10.0,734.0,8.174629,2.0,2.0,2.0,0.0,medium,2,0.0,0
1,client_65de48885f4ef01b,content_62673eea26c31c17,6.0,24542.0,6.187605,29.0,24844.0,5.392198,8.0,7759.0,7.866490,2.0,2.0,2.0,0.0,high,4,0.0,0
2,client_65de48885f4ef01b,content_4c185d1c173cd53d,3.0,113.0,10.719087,3.0,43.0,11.539352,3.0,122.0,9.395247,3.0,3.0,2.0,0.0,medium,2,0.0,0
3,client_65de48885f4ef01b,content_bd07be40ea0d5f54,0.0,76.0,4.881643,0.0,19.0,6.135417,0.0,147.0,33.655578,2.0,2.0,4.0,0.0,medium,2,0.0,1
4,client_65de48885f4ef01b,content_40e28f4b41764012,2.0,298.0,8.006499,3.0,118.0,3.528997,1.0,81.0,4.553223,2.0,2.0,2.0,0.0,medium,2,0.0,0


In [5]:
model_df["click_change_w1_w2"] = model_df["week2_clicks"] - model_df["week1_clicks"]
model_df["impression_change_w1_w2"] = model_df["week2_impressions"] - model_df["week1_impressions"]
model_df["log_w2_impressions"] = np.log1p(model_df["week2_impressions"])
model_df["log_w2_clicks"] = np.log1p(model_df["week2_clicks"])
model_df["w2_ctr"] = np.where(model_df["week2_impressions"] > 0, (model_df["week2_clicks"] / model_df["week2_impressions"]) * 100, 0.0)

feature_columns = [
    "week1_impressions", "week2_impressions", "week1_avg_position", "week2_avg_position",
    "w1_tier", "w2_tier", "historical_tier_change",
    "click_change_w1_w2", "impression_change_w1_w2",
    "log_w2_impressions", "log_w2_clicks", "w2_ctr"
]

model_df = model_df.reset_index(drop=True)

X = model_df[feature_columns].fillna(0)
y = model_df["future_decline"].values
clients = model_df["client_hash_id"].values
contents = model_df["content_hash_id"].values
baseline_scores = model_df["baseline_score"].values

print("=== Verification of Array Shapes ===")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"clients shape: {clients.shape}")
print(f"contents shape: {contents.shape}")
print(f"baseline_scores shape: {baseline_scores.shape}")

=== Verification of Array Shapes ===
X shape: (13531, 12)
y shape: (13531,)
clients shape: (13531,)
contents shape: (13531,)
baseline_scores shape: (13531,)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

To evaluate my model fairly against my baseline rule, I will adhere to my comparison contract:
1. Every model and baseline rule will be evaluated across the same client-holdout folds (`GroupKFold` on `client_hash_id`).
2. My primary metric will be Precision@50, which evaluates the proportion of actually declining pages in the top 50 queue places. I will also track Precision@20 and Precision@100 to see how scalable my models are.
3. For each test fold, I will record the percentage of pages that actually declined. This will provide a reference point that I can use to compare the Precision@K results against each other

4. Score ties will be broken by ordering in the ranked queue by increasing content_hash_id alphabetical values

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, content_ids, k):
    eval_df = pd.DataFrame({
        'y': np.asarray(y_true, dtype=int),
        'score': np.asarray(scores, dtype=float),
        'content_hash_id': np.asarray(content_ids, dtype=str)
    })


    sorted_df = eval_df.sort_values(
        by=['score', 'content_hash_id'],
        ascending=[False, True]
    )

    top_k_labels = sorted_df.head(k)['y']
    return float(top_k_labels.mean())

contents = model_df["content_hash_id"].values


In [7]:

# 2. Initialize 5-Fold GroupKFold on client_hash_id
gkf = GroupKFold(n_splits=5)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=clients), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    contents_test = contents[test_idx]
    base_scores_test = baseline_scores[test_idx]
    base_rate_fold = y_test.mean()

    # Baseline rule
    fold_results.append({
        'fold': fold, 'model': 'Custom Baseline (baseline_score)', 'base_rate': base_rate_fold,
        'p20': precision_at_k(y_test, base_scores_test, contents_test, 20),
        'p50': precision_at_k(y_test, base_scores_test, contents_test, 50),
        'p100': precision_at_k(y_test, base_scores_test, contents_test, 100)
    })

    # Logistic Regression
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(X_train_s, y_train)
    p_lr = lr.predict_proba(X_test_s)[:, 1]

    fold_results.append({
        'fold': fold, 'model': 'Logistic Regression', 'base_rate': base_rate_fold,
        'p20': precision_at_k(y_test, p_lr, contents_test, 20),
        'p50': precision_at_k(y_test, p_lr, contents_test, 50),
        'p100': precision_at_k(y_test, p_lr, contents_test, 100)
    })

    # Decision Tree
    dt = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
    dt.fit(X_train, y_train)
    p_dt = dt.predict_proba(X_test)[:, 1]

    fold_results.append({
        'fold': fold, 'model': 'Decision Tree (d=4)', 'base_rate': base_rate_fold,
        'p20': precision_at_k(y_test, p_dt, contents_test, 20),
        'p50': precision_at_k(y_test, p_dt, contents_test, 50),
        'p100': precision_at_k(y_test, p_dt, contents_test, 100)
    })

    # Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    p_rf = rf.predict_proba(X_test)[:, 1]

    fold_results.append({
        'fold': fold, 'model': 'Random Forest (n=100, d=8)', 'base_rate': base_rate_fold,
        'p20': precision_at_k(y_test, p_rf, contents_test, 20),
        'p50': precision_at_k(y_test, p_rf, contents_test, 50),
        'p100': precision_at_k(y_test, p_rf, contents_test, 100)
    })

fold_df = pd.DataFrame(fold_results)


In [8]:
# Summary table
summary = fold_df.groupby('model').agg({
    'base_rate': 'mean',
    'p20': ['mean', 'std'],
    'p50': ['mean', 'std'],
    'p100': ['mean', 'std']
}).reset_index()

summary.columns = ['Model', 'Base Rate', 'P@20 Mean', 'P@20 Std',
                   'P@50 Mean', 'P@50 Std', 'P@100 Mean', 'P@100 Std']

print("5-Fold GroupKFold Cross-Validation Summary vs Custom Baseline Score")
display(summary[['Model', 'Base Rate', 'P@20 Mean', 'P@50 Mean', 'P@100 Mean']].round(4))

5-Fold GroupKFold Cross-Validation Summary vs Custom Baseline Score


,Model,Base Rate,P@20 Mean,P@50 Mean,P@100 Mean
0,Custom Baseline (baseline_score),0.2041,0.88,0.816,0.750
1,Decision Tree (d=4),0.2041,0.82,0.844,0.822
2,Logistic Regression,0.2041,0.84,0.820,0.826
3,"Random Forest (n=100, d=8)",0.2041,0.94,0.880,0.866


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

#### Features

My models use ranking and performance signals from the first half of March 2026 as their features, including historical_tier_change, w2_tier and w1_tier and changes in clicks and impressions. These features decribe the performance of each page leading up to 15th March 2026.



#### 2. Error Analysis:
* **False Positives**: Pages that suffered a rank tier drop in Week 2, but regained their higher rankings in Late March (rank turbulence rather than true content decay).
* **False Negatives**: Pages that appeared perfectly stable during Weeks 1 and 2, but dropped unexpectedly in Late March. This could be due to external or seasonal changes that could not be predicted from early march data.

#### 3. Model Conclusion
* **Custom Baseline Score**: Evaluates Week 1 vs. Week 2 drops against Late March outcomes.
* **Random Forest**: By combining multi-feature interaction signals (such as position tier boundaries alongside click changes), my Random Forest consistently achieves the highest mean Precision@50 out of all my models without data leakage.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.